# RTD Conversational Agent

This notebook provides an interactive interface for running and testing the agent.

In [1]:
# Auto-reload modules when they change
%load_ext autoreload
%autoreload 2

In [2]:
# Import agent components
from agent import (
    agent_context,
    classify_intent,
    process_user_request,
    run,
    set_dataframe,
)

## Option 1: Start MQTT Loop

Run this cell to start listening for MQTT messages from Unity.

In [ ]:
# Start the MQTT message loop (blocks until Ctrl+C)
run()

Connecting to 8e66e8b014bd4447ba51ade00be22032.s1.eu.hivemq.cloud:8883...
Starting message loop. Press Ctrl+C to exit.
Connected to MQTT broker
Subscribed to 'agent_in'

Received: {
  "chart_metadata_index": {
    "chart_count": 4,
    "charts": [
      {
        "chart_id": 1,
        "data_name": "language",
        "chart_type": "speakers",
        "columns": [
          "ye...
Chart metadata index registered

Received: {
  "user_request_for_agent": {
    "transcript": {
      "text_transcript": "load the computer part price chart",
      "confidence": 0.99
    },
    "touchdata": {
      "left_touch": "No left touch...
Detected Intent: load_chart | Deictic: False
Sent response to 'agent_out'

Received: {
  "rtd_data_for_agent": {
    "chart_type": "price",
    "data_name": "ram",
    "schema": {
      "$schema": "https://vega.github.io/schema/vega-lite/v5.json",
      "description": "Computer compon...
RTD data registered: chart_type=price, data_name=ram

Received: {
  "message_typ

## Option 2: Test Components Individually

Use these cells to test individual components.

**Note:** If you ran Option 1 first, skip Cell 7 to keep the real Unity data. Cell 7 overwrites context with test data.

In [3]:
# Test intent classification
test_queries = [
    "Show me the rainfall data",
    "What is the trend?",
    "What is this point?",
    "Zoom to 2020",
    "What was the average rainfall?",
]

for q in test_queries:
    intent = classify_intent(q)
    print(f"{q!r} -> {intent}")

'Show me the rainfall data' -> load_chart
'What is the trend?' -> trend
'What is this point?' -> touch_interaction
'Zoom to 2020' -> operations
'What was the average rainfall?' -> data_analysis


In [4]:
# Load test data
import pandas as pd

# Example: create test DataFrame
test_df = pd.DataFrame({
    "Year": [2019, 2020, 2021, 2022, 2023],
    "Rainfall": [450, 520, 380, 410, 490]
})

set_dataframe(test_df)
agent_context["x_field"] = "Year"
agent_context["y_field"] = "Rainfall"
agent_context["chart_type"] = "line"
agent_context["graph_thread_id"] = "test-1"

print("Test data loaded:")
print(test_df)

✅ DataFrame registered with columns: ['Year', 'Rainfall']
Test data loaded:
   Year  Rainfall
0  2019       450
1  2020       520
2  2021       380
3  2022       410
4  2023       490


In [5]:
# Test a full request (simulating MQTT message)
import json

test_request = json.dumps({
    "user_request_for_agent": {
        "transcript": {
            "text_transcript": "What year had the highest rainfall?"
        },
        "touchdata": {},
        "highlighted_context": {}
    }
})

result = process_user_request(test_request)
print("Response:", result["response"])

🔎 Detected Intent: data_analysis | Deictic: False
⏱️ Streaming completed in 2.78 seconds.
extracted nodes: {'node_1': {'value': 2020.0}, 'node_2': {'value': 520.0}, 'node_3': {'year': 2020}}
Response: The year with the highest rainfall was 2020, with a total of 520 mm of rainfall.


In [6]:
# Check current agent context
for key, value in agent_context.items():
    if key != "df":  # Skip DataFrame for cleaner output
        print(f"{key}: {value}")

df_columns: ['Year', 'Rainfall']
x_field: Year
y_field: Rainfall
chart_type: line
graph_thread_id: test-1
last_intent: data_analysis
